In [4]:
import pandas as pd

infections_df = pd.read_csv("../Data/cleaned/healthcare_infections_cleaned.csv")
print("Shape:", infections_df.shape)
print("\nColumns:")
for col in infections_df.columns:
    print(f"  {col}")

Shape: (172404, 14)

Columns:
  facility_id
  facility_name
  address
  citytown
  state
  zip_code
  countyparish
  telephone_number
  measure_id
  measure_name
  compared_to_national
  score
  start_date
  end_date


In [5]:
print("unique measures:", infections_df["measure_id"].nunique())
print("\nmeasure list:")
print(infections_df[["measure_id", "measure_name"]].drop_duplicates().to_string())

unique measures: 36

measure list:
         measure_id                                                                                        measure_name
0     HAI_1_CILOWER          Central Line Associated Bloodstream Infection (ICU + select Wards): Lower Confidence Limit
1     HAI_1_CIUPPER          Central Line Associated Bloodstream Infection (ICU + select Wards): Upper Confidence Limit
2        HAI_1_DOPC                                Central Line Associated Bloodstream Infection: Number of Device Days
3   HAI_1_ELIGCASES                 Central Line Associated Bloodstream Infection (ICU + select Wards): Predicted Cases
4   HAI_1_NUMERATOR                  Central Line Associated Bloodstream Infection (ICU + select Wards): Observed Cases
5         HAI_1_SIR                                  Central Line Associated Bloodstream Infection (ICU + select Wards)
6     HAI_2_CILOWER           Catheter Associated Urinary Tract Infections (ICU + select Wards): Lower Confidence Limit
7    

In [6]:
sir_measures = infections_df[infections_df["measure_id"].str.endswith("_SIR")]

print("rows after filter:", len(sir_measures))
print("measures kept:", sir_measures["measure_id"].unique())

rows after filter: 28734
measures kept: ['HAI_1_SIR' 'HAI_2_SIR' 'HAI_3_SIR' 'HAI_4_SIR' 'HAI_5_SIR' 'HAI_6_SIR']


In [7]:
sir_measures = sir_measures.copy()
sir_measures["score"] = sir_measures["score"].replace("Not Available", None)
sir_measures["score"] = pd.to_numeric(sir_measures["score"])

infections_pivot = sir_measures.pivot_table(
    index="facility_id",
    columns="measure_id",
    values="score",
    aggfunc="first"
).reset_index()

infections_pivot.columns.name = None

hospital_info = sir_measures[["facility_id", "facility_name", "address", 
                               "citytown", "state", "zip_code", 
                               "countyparish", "telephone_number"]].drop_duplicates("facility_id")

df_final = hospital_info.merge(infections_pivot, on="facility_id", how="outer")

print("final shape:", df_final.shape)
print("duplicate facility_ids:", df_final["facility_id"].duplicated().sum())
df_final.head(2)

final shape: (4789, 14)
duplicate facility_ids: 0


,facility_id,facility_name,address,citytown,state,zip_code,countyparish,telephone_number,HAI_1_SIR,HAI_2_SIR,HAI_3_SIR,HAI_4_SIR,HAI_5_SIR,HAI_6_SIR
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,0.418,0.125,0.292,NaN,0.390,0.395
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,1.430,1.054,0.804,NaN,2.431,0.430
